# A/B Testing Toolkit: Walkthrough

This notebook works through two examples:

1. CTR test (binary)
2. Revenue per user test (continuous)

Each example shows the frequentist and Bayesian view side by side.

In [ ]:
import sys
sys.path.append('..')

from src import (two_proportion_ztest, welch_ttest,
                 prob_b_beats_a, credible_interval, expected_loss,
                 proportion_sample_size, continuous_sample_size)
from src.sim import generate_binary, generate_revenue

## Example 1: CTR test

Baseline CTR is 5%, we want to detect a 0.5pp lift at 80% power, alpha=0.05.

In [ ]:
n = proportion_sample_size(baseline_rate=0.05, mde=0.005, power=0.8, alpha=0.05)
print(f'per-arm n = {n}')

In [ ]:
df = generate_binary(n_per_arm=n, p_a=0.05, p_b=0.056, seed=7)
df.groupby('variant')['converted'].agg(['sum', 'count', 'mean'])

### Frequentist

In [ ]:
g = df.groupby('variant')['converted']
succ_a, n_a = int(g.sum()['A']), int(g.count()['A'])
succ_b, n_b = int(g.sum()['B']), int(g.count()['B'])
two_proportion_ztest(succ_a, n_a, succ_b, n_b)

### Bayesian

In [ ]:
p_b_beats_a = prob_b_beats_a(succ_a, n_a, succ_b, n_b, seed=0)
loss_a, loss_b = expected_loss(succ_a, n_a, succ_b, n_b, seed=0)
print(f'P(B>A): {p_b_beats_a:.3f}')
print(f'expected loss A: {loss_a:.5f}, B: {loss_b:.5f}')
print(f'95% CI A: {credible_interval(succ_a, n_a)}')
print(f'95% CI B: {credible_interval(succ_b, n_b)}')

## Example 2: revenue per user

Mean of 12 USD, sd of 8.  We want to spot a 0.6 USD lift.

In [ ]:
n_rev = continuous_sample_size(mde=0.6, sd=8.0, power=0.8, alpha=0.05)
print(f'per-arm n = {n_rev}')
rev = generate_revenue(n_per_arm=n_rev, mean_a=12.0, mean_b=12.6, sd=8.0, seed=11)
rev.groupby('variant')['revenue'].agg(['mean', 'std', 'count'])

In [ ]:
a = rev[rev.variant == 'A']['revenue'].values
b = rev[rev.variant == 'B']['revenue'].values
welch_ttest(a, b)